In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
dx_df = pd.read_csv('data/DXSUM_17Feb2026.csv')
dem  = pd.read_csv('data/PTDEMOG_17Feb2026.csv') 
adas = pd.read_csv('data/ADAS_17Feb2026.csv')
medhist = pd.read_csv('data/MEDHIST_17Feb2026.csv')
recmhist = pd.read_csv('data/RECMHIST_17Feb2026.csv')
demographics = pd.read_csv('data/PTDEMOG_17Feb2026.csv')
apoe_df = pd.read_csv("data/APOERES_17Feb2026.csv")
apoe_df["CARRIER"] = apoe_df["GENOTYPE"].isin(["2/4","3/4","4/4"])
apoe_df["HOMO"] = apoe_df["GENOTYPE"].isin(["4/4"])

In [3]:
def visit_to_months(v):
    if pd.isna(v):
        return np.nan
    v = v.lower()
    if v in ["bl", "sc"]:
        return 0
    if v.startswith("m"):
        return float(v[1:])  # strip the 'm'
    if v.startswith("v"):
        return float(v[1:])  # strip the 'v'
    
    return np.nan

baseline_codes = ["bl", "4_bl"]
adas["time_months"] = adas["VISCODE"].apply(visit_to_months)

baseline_dates = (
    adas[adas["VISCODE"].isin(baseline_codes)]
    .groupby("RID")["VISDATE"]
    .min()
)

adas_df = adas.merge(
    baseline_dates.rename("baseline_date"),
    on="RID",
    how="left"
)

adas_df["days_since_bl"] = (
    pd.to_datetime(adas_df["VISDATE"]) -
    pd.to_datetime(adas_df["baseline_date"])
)

In [4]:
# Merge ADAS scores with diagnosis data
adas_AD_dx = pd.merge(adas_df, dx_df[["RID", "VISCODE", "DIAGNOSIS"]], on=["RID", "VISCODE"], how="left")
adas_AD_dx['VISDATE'] = pd.to_datetime(adas_AD_dx['VISDATE'])


### Finding comorbidities with string matching:

In [5]:
UTI_KEYWORDS = [
    "urinary tract infection",
    "uti",
    "bladder infection",
    "cystitis",
    "urosepsis",
    "pyelonephritis",
    "urinary infection",
    "recurrent uti"
]

Z867_KEYWORDS = [
    "history of stroke",
    "history of tia",
    "prior stroke",
    "previous stroke",
    "prior tia",
    "cva",
    "transient ischemic attack",
    "history of myocardial infarction",
    "previous mi",
    "prior mi",
    "ischemic heart disease",
    "coronary artery disease",
    "cad",
    "peripheral vascular disease",
    "pvd",
    "vascular disease"
]

Z864_KEYWORDS = [
    "alcohol abuse",
    "alcohol dependence",
    "alcoholism",
    "substance abuse",
    "drug abuse",
    "drug dependence",
    "opioid dependence",
    "cocaine use",
    "heroin use",
    "illicit drug use",
    "history of substance abuse"
]

SLEEP_KEYWORDS = [
    "insomnia",
    "sleep disorder",
    "sleep disturbance",
    "poor sleep",
    "sleep apnea",
    "obstructive sleep apnea",
    "osa",
    "hypersomnia",
    "sleep fragmentation",
    "restless legs"
]

DENTAL_KEYWORDS = [
    "periodontitis",
    "gingivitis",
    "gum disease",
    "dental abscess",
    "tooth infection",
    "tooth loss",
    "edentulous",
    "poor dentition",
    "dental caries",
    "tooth decay"
]

In [6]:
SPECIFIC_KEYWORDS = {
    "UTI": UTI_KEYWORDS,
    "Z867": Z867_KEYWORDS,
    "Z864": Z864_KEYWORDS,
    "Sleep": SLEEP_KEYWORDS,
    "Dental": DENTAL_KEYWORDS
}

def classify_specific(desc):
    if pd.isna(desc):
        return None
    
    d = desc.lower()
    d = d.replace("hx of", "history of")
    d = d.replace("h/o", "history of")
    
    for category, keywords in SPECIFIC_KEYWORDS.items():
        if any(k in d for k in keywords):
            return category
    
    return None

recmhist["specific_flag"] = recmhist["MHDESC"].apply(classify_specific)

In [38]:
CNS_KEYWORDS = {
    'cerebrovascular': ['stroke', 'cerebrovascular', 'tia', 'transient ischemic', 'cva'],
    'insomnia': ['insomnia', 'sleep disorder'],
    'anxiety': ['anxiety', 'anxious'],
    'depression': ['depression', 'depressive', 'depressed'],
    'head_injury': ['head injury', 'tbi', 'traumatic brain', 'concussion', 'head trauma']
}

PERIPHERAL_KEYWORDS = {
    'hypertension': ['hypertension', 'high blood pressure', 'htn'],
    'hyperlipidemia': ['hyperlipidemia', 'hypercholesterolemia', 'high cholesterol', 'dyslipidemia'],
    'diabetes': ['diabetes', 'diabetic', 'dm', 'niddm', 'iddm'],
    'atrial_fibrillation': ['atrial fibrillation', 'afib', 'a fib', 'a-fib'],
    'coronary': ['coronary', 'ischemic heart', 'ihd', 'cad', 'coronary artery', 'myocardial infarction', 'mi', 'heart attack'],
    'anemia': ['anemia', 'anaemia'],
    'hypothyroid': ['hypothyroid', 'thyroid'],
    'skin_inflammatory': ['psoriasis', 'eczema', 'dermatitis', 'skin inflammation'],
    'pulmonary': ['copd', 'asthma', 'pulmonary', 'emphysema', 'chronic bronchitis', 'respiratory', 'lung disease'],
    'kidney': ['kidney', 'renal', 'ckd', 'chronic kidney'],
    'hepatitis': ['hepatitis', 'liver disease', 'cirrhosis'],
    'osteoporosis': ['osteoporosis', 'bone density'],
    'hearing_loss': ['hearing loss', 'deaf', 'hearing impair'],
    'cancer': ['cancer', 'malignancy', 'carcinoma', 'tumor', 'neoplasm'],
    'gastrointestinal': ['gastro', 'ulcer', 'reflux', 'gerd', 'ibs', 'crohn', 'colitis', 'diverticulitis'],
    'cataract': ['cataract'],
}

def classify_condition_detailed(desc):
    """
    Classify condition into CNS, Peripheral, or Other
    Returns tuple: (category, specific_condition)
    """
    if pd.isna(desc):
        return None, None
    
    d = desc.lower()
    
    # Check CNS conditions
    for condition, keywords in CNS_KEYWORDS.items():
        if any(k in d for k in keywords):
            return "CNS", condition
    
    # Check Peripheral conditions
    for condition, keywords in PERIPHERAL_KEYWORDS.items():
        if any(k in d for k in keywords):
            return "Peripheral", condition
    
    return "Other", None

recmhist["comorb_category"] = recmhist["MHDESC"].apply(
    lambda x: classify_condition_detailed(x)[0]
)
recmhist["specific_condition"] = recmhist["MHDESC"].apply(
    lambda x: classify_condition_detailed(x)[1]
)

In [39]:
comorb_counts = (
    recmhist[recmhist["comorb_category"].isin(["CNS", "Peripheral"])]
    .groupby(["RID", "VISCODE", "comorb_category", "specific_condition"])
    .size()
    .reset_index(name="count")
    .groupby(["RID", "VISCODE", "comorb_category"])
    .agg(
        n_conditions=("specific_condition", "nunique"),
        total_entries=("count", "sum")
    )
    .reset_index()
)

# Create wide format
comorb_wide = comorb_counts.pivot_table(
    index=["RID", "VISCODE"],
    columns="comorb_category",
    values="n_conditions",
    fill_value=0
).reset_index()

# Calculate total multimorbidity burden
comorb_wide["total_conditions"] = (
    comorb_wide.get("CNS", 0) + 
    comorb_wide.get("Peripheral", 0)
)

# Categorize burden as in the paper
def categorize_burden(n):
    if n <= 2:
        return "Low"
    elif n <= 5:
        return "Medium"
    else:
        return "High"

comorb_wide["burden_category"] = comorb_wide["total_conditions"].apply(categorize_burden)

# For CNS burden: 0 vs 1+ 
comorb_wide["CNS_burden"] = comorb_wide["CNS"].apply(lambda x: "0" if x == 0 else "1+")

# For Peripheral burden: 0-1 vs 2+ 
comorb_wide["Peripheral_burden"] = comorb_wide["Peripheral"].apply(
    lambda x: "0-1" if x <= 1 else "2+"
)

In [40]:
specific_flags_visit = (
    recmhist[recmhist["specific_flag"].notna()]
    .assign(flag=1)
    .pivot_table(
        index=["RID", "VISCODE"],
        columns="specific_flag",
        values="flag",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

comorb_wide_new = comorb_wide.merge(
    specific_flags_visit,
    on=["RID", "VISCODE"],
    how="left"
)

# Replace NaNs with 0s for the specific flags
for col in ["UTI", "Z867", "Z864", "Sleep", "Dental"]:
    if col in comorb_wide_new.columns:
        comorb_wide_new[col] = comorb_wide_new[col].fillna(0)

In [41]:
AD_dem = adas_AD_dx.merge(
    demographics[["RID", "PTDOB", 'PTGENDER', 'PTEDUCAT']],
    on="RID",
    how="left"
)
print(f"Number of individuals with demographic data and cognitive data: {AD_dem['RID'].nunique()}")

AD = AD_dem.merge(
    comorb_wide_new[["RID", "total_conditions", "CNS", "Peripheral", "UTI", "Z867", "Z864", "Sleep", "Dental"]],
    on="RID",
    how="inner")
print(f"Number of individuals with comorbidity data: {AD['RID'].nunique()}")

Number of individuals with demographic data and cognitive data: 3027
Number of individuals with comorbidity data: 1681


In [45]:
ad_start = AD[['RID', 'VISDATE', 'TOTAL13', 'PTGENDER', 'PTDOB', 'PTEDUCAT']].copy()
first_entries = (
    ad_start
    .sort_values('VISDATE')
    .groupby('RID')
    .first()
    .reset_index()
)

first_entries['VISDATE'] = pd.to_datetime(first_entries['VISDATE'])
first_entries['PTDOB'] = pd.to_datetime(first_entries['PTDOB'])
# Calculate age at conversion (in years)
first_entries['age_at_baseline'] = (
    (first_entries['VISDATE'] - first_entries['PTDOB'])
    .dt.days / 365.25
)

ad_start = first_entries.rename(columns={
    'VISDATE': 'Study_start_date',
    'TOTAL13': 'TOTAL13_AD_start'
})

AD_new = AD.merge(ad_start[['RID', 'Study_start_date', 'TOTAL13_AD_start', 'age_at_baseline']], on='RID', how='left')


/var/folders/g_/qzn_bmsd7v9fwp049f83fwtr0000gp/T/ipykernel_86717/939935225.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  first_entries['PTDOB'] = pd.to_datetime(first_entries['PTDOB'])


In [46]:
model_vars = [
    "TOTAL13",
    "TOTAL13_AD_start",
    "VISCODE",
    "VISDATE",
    "PHASE",
    "days_since_entry",
    "age_at_baseline",
    "PTGENDER",
    "CARRIER",
    "HOMO",
    "PTEDUCAT",
    "CNS",
    "total_conditions",
    "Peripheral",
    "RID",
    "UTI", "Z867", "Z864", "Sleep", "Dental"
]

In [47]:
AD_df = AD_new.merge(apoe_df[["RID", "CARRIER", "HOMO"]], on="RID", how="inner")
    
AD_df['VISDATE'] = pd.to_datetime(AD_df['VISDATE'])
AD_df['Study_start_date'] = pd.to_datetime(AD_df['Study_start_date'])
AD_df['days_since_entry'] = (AD_df['VISDATE'] - AD_df['Study_start_date']).dt.days

AD_no_EO = AD_df[AD_df['age_at_baseline'] >= 65]

AD_model = AD_no_EO[model_vars].dropna()

AD_model["age_c"] = AD_model["age_at_baseline"] - AD_no_EO["age_at_baseline"].mean()
AD_model["edu_c"] = AD_model["PTEDUCAT"] - AD_no_EO["PTEDUCAT"].mean()
AD_model["time_years"] = AD_model["days_since_entry"] / 365.25

print(AD_model["RID"].nunique())
print(AD_model.head())

1476
   TOTAL13  TOTAL13_AD_start VISCODE    VISDATE  PHASE  days_since_entry  \
3    17.00             13.67     m06 2008-03-25  ADNI1             230.0   
4    13.67             13.67      bl 2007-08-08  ADNI1               0.0   
5     2.00              4.00     m24 2009-10-30  ADNI1             821.0   
6     2.00              4.00     m24 2009-10-30  ADNI1             821.0   
7     2.00              4.00     m24 2009-10-30  ADNI1             821.0   

   age_at_baseline  PTGENDER  CARRIER   HOMO  ...  Peripheral   RID  UTI  \
3        74.269678       1.0    False  False  ...         4.0  1411  0.0   
4        74.269678       1.0    False  False  ...         4.0  1411  0.0   
5        71.581109       1.0    False  False  ...         4.0  1408  0.0   
6        71.581109       1.0    False  False  ...         3.0  1408  0.0   
7        71.581109       1.0    False  False  ...         3.0  1408  1.0   

   Z867  Z864  Sleep  Dental     age_c     edu_c  time_years  
3   1.0   0.0    0

### Add in CSF data:

In [49]:
CSF = pd.read_csv("data/UPENNBIOMK_ROCHE_ELECSYS_19Feb2026.csv")

CSF["PTAU_ABETA42"] = CSF["PTAU"] / CSF["ABETA42"]
CSF["PT_AB_std"] = (CSF["PTAU_ABETA42"] - CSF["PTAU_ABETA42"].mean()) / CSF["PTAU_ABETA42"].std()

CSF_bl = CSF[CSF["VISCODE2"] == "bl"]
CSF_bl["CSF_date"] = CSF_bl["EXAMDATE"]

/var/folders/g_/qzn_bmsd7v9fwp049f83fwtr0000gp/T/ipykernel_86717/375718845.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CSF_bl["CSF_date"] = CSF_bl["EXAMDATE"]


In [50]:
CSF_AD = AD_model.merge(
    CSF_bl[["RID", "VISCODE2", "ABETA42", "TAU", "PTAU", "PTAU_ABETA42", "PT_AB_std", "CSF_date"]],
    on=["RID"],
    how="inner")
CSF_AD["time_sq"] = CSF_AD["time_years"] ** 2
CSF_AD = CSF_AD.dropna(subset=["TOTAL13"])
CSF_AD = CSF_AD.dropna(subset=["PTAU_ABETA42"])
CSF_AD["baseline_c"] = (
CSF_AD["TOTAL13_AD_start"] - CSF_AD["TOTAL13_AD_start"].mean())
print(CSF_AD["RID"].nunique())

1020


In [52]:
CSF_AD.to_csv('CSF_AD.csv', index=False)